# UAW-AFS Reconstruction Validation

This notebook tests whether BNN uncertainty-aware frequency sampling plus vector fitting reconstructs full BCLF responses better than Uniform or Random frequency sampling. It is an offline precondition before wiring UAW-AFS into online optimization.

The active mode is `smoke` by default so the notebook can be run quickly. Switch `RUN_MODE` to `full` for the planned validation settings.

In [ ]:
from pathlib import Path
import json
import pickle
import sys
import time

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from modeling import BCLFResponseDataset, LegacyBNNResponseSurrogate, response_db
from optimization import BCLFOptimizationSpec
from src.poles_init import basic_poles_initialize
from src.vector_fitting import vector_fit, evaluate
from src.utils import frequency_to_s

OUTPUT_DIR = REPO_ROOT / 'experiments' / 'uaw_afs_validation'
DATASET_PATH = REPO_ROOT / 'data' / 'filter_bnn_online_uaw_uaw_dataset.pkl'
VARIABLE_ORDER = ['L', 'L1', 'L2', 'W', 'W1', 'W2', 'S1', 'S2']
FREQS = np.arange(3.5e9, 8.713e9, 0.013e9, dtype=float)

FULL_CONFIG = {
    'seeds': [0, 1, 2, 3, 4],
    'train_size': 30,
    'eval_size': 50,
    'freq_sample_size': 20,
    'hidden_dims': [256, 256, 256],
    'epochs': 800,
    'batch_size': 1024,
    'lr': 1e-3,
    'bayesian_loss_weight': 1e-4,
    'mc_samples': 64,
    'device': 'cuda',
}
SMOKE_OVERRIDES = {
    'seeds': [0],
    'train_size': 8,
    'eval_size': 5,
    'freq_sample_size': 20,
    'hidden_dims': [16],
    'epochs': 1,
    'batch_size': 256,
    'mc_samples': 4,
    'device': 'cpu',
}
RUN_MODE = 'smoke'  # change to 'full' for the planned validation run

CONFIG = dict(FULL_CONFIG)
if RUN_MODE == 'smoke':
    CONFIG.update(SMOKE_OVERRIDES)
elif RUN_MODE != 'full':
    raise ValueError("RUN_MODE must be 'smoke' or 'full'")

CONFIG

## Load BCLF Dataset

In [ ]:
def load_online_pickle_as_response_dataset(path: Path) -> BCLFResponseDataset:
    with path.open('rb') as fh:
        payload = pickle.load(fh)
    samples = payload['dataset'] if isinstance(payload, dict) else payload
    X = np.asarray([item['geo_params'] for item in samples], dtype=float)
    s_params = np.asarray([item['s_params'] for item in samples], dtype=complex)
    case_ids = [str(item.get('caseid', idx)) for idx, item in enumerate(samples)]
    return BCLFResponseDataset(
        variable_order=list(VARIABLE_ORDER),
        freqs=FREQS,
        X=X,
        s_params=s_params,
        filenames=[f'{case_id}.s2p' for case_id in case_ids],
        keys=case_ids,
        case_ids=case_ids,
    )

dataset = load_online_pickle_as_response_dataset(DATASET_PATH)
print(dataset.n_samples, dataset.n_freqs, dataset.s_params.shape)
assert dataset.n_freqs == dataset.s_params.shape[-1]

## Sampling And Reconstruction Helpers

All methods include the first and last frequency points, then select the remaining interior points. This makes vector fitting stable while keeping the comparison fair across UAW, Uniform, and Random.

In [ ]:
SPEC = BCLFOptimizationSpec(
    passband_ghz=(4.8, 5.4),
    lower_stopband_ghz=(3.6, 4.4),
    upper_stopband_ghz=(5.8, 8.7),
    min_passband_s21_db=-5.0,
    max_passband_s11_db=-7.0,
    max_stopband_s21_db=-15.0,
    s11_weight=0.5,
    stopband_weight=0.5,
)


def db_mae(true_s, pred_s, row, col):
    return float(np.mean(np.abs(response_db(true_s[None, ...], row, col)[0] - response_db(pred_s[None, ...], row, col)[0])))


def objective_value(s_params):
    return float(SPEC.evaluate(dataset.freqs, s_params).scalar_objective())


def objective_abs_error(true_s, pred_s):
    return abs(objective_value(true_s) - objective_value(pred_s))


def select_frequency_indices(method, uncertainty_profile, n_points, rng):
    n_freqs = len(dataset.freqs)
    if n_points > n_freqs:
        raise ValueError('freq_sample_size exceeds available frequencies')
    if n_points <= 2:
        return np.linspace(0, n_freqs - 1, n_points, dtype=int)

    interior = np.arange(1, n_freqs - 1, dtype=int)
    need = n_points - 2
    if method == 'uniform':
        selected = np.linspace(1, n_freqs - 2, need, dtype=float)
        interior_idx = np.unique(np.round(selected).astype(int))
        while len(interior_idx) < need:
            missing = np.setdiff1d(interior, interior_idx, assume_unique=False)
            interior_idx = np.sort(np.append(interior_idx, missing[: need - len(interior_idx)]))
    elif method == 'random':
        interior_idx = np.sort(rng.choice(interior, size=need, replace=False))
    elif method == 'uaw':
        u = np.asarray(uncertainty_profile, dtype=float).reshape(-1)
        u = np.maximum(u, 0.0)
        weights = u[interior]
        if not np.isfinite(weights).all() or float(weights.sum()) <= 0.0:
            weights = np.ones_like(weights, dtype=float)
        else:
            weights = 0.1 * np.ones_like(weights, dtype=float) / len(weights) + 0.9 * weights / float(weights.sum())
        interior_idx = np.sort(rng.choice(interior, size=need, replace=False, p=weights / weights.sum()))
    else:
        raise ValueError(f'unknown method: {method}')
    return np.asarray([0, *interior_idx.tolist(), n_freqs - 1], dtype=int)


def fit_channel_from_indices(s_params, indices, row, col, n_poles_real=4, n_poles_complex=8):
    sample_freqs = dataset.freqs[indices]
    values = s_params[row, col, indices]
    init_poles = basic_poles_initialize(
        freqs_range=(float(dataset.freqs[0]), float(dataset.freqs[-1])),
        n_poles_real=n_poles_real,
        n_poles_complex=n_poles_complex,
    )
    poles, residues, d, h = vector_fit(
        sample_freqs,
        values,
        init_poles=init_poles,
        max_iter=30,
        tol=1e-7,
        atol=1e-7,
        rtol=1e-7,
    )
    return evaluate(poles, residues, d, h, frequency_to_s(dataset.freqs))


def reconstruct_sparams(s_params, indices):
    pred = np.zeros_like(s_params, dtype=complex)
    pred[0, 0, :] = fit_channel_from_indices(s_params, indices, 0, 0)
    pred[0, 1, :] = fit_channel_from_indices(s_params, indices, 0, 1)
    pred[1, 0, :] = pred[0, 1, :]
    pred[1, 1, :] = pred[0, 0, :]
    return pred


def uncertainty_profile_from_samples(response_samples, local_pos):
    s11 = response_db(response_samples[:, local_pos:local_pos + 1], 0, 0)[:, 0, :]
    s21 = response_db(response_samples[:, local_pos:local_pos + 1], 0, 1)[:, 0, :]
    return np.var(s11, axis=0) + np.var(s21, axis=0)

## Run Validation

This cell writes CSV outputs under `experiments/uaw_afs_validation/`. In smoke mode it runs only one seed and five evaluation samples.

In [ ]:
def run_validation(config):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    rows = []
    config_to_save = {'run_mode': RUN_MODE, **config}
    (OUTPUT_DIR / 'run_config.json').write_text(json.dumps(config_to_save, indent=2), encoding='utf-8')

    for seed in config['seeds']:
        seed = int(seed)
        rng = np.random.default_rng(seed)
        order = rng.permutation(dataset.n_samples)
        train_idx = np.sort(order[: int(config['train_size'])])
        eval_idx = np.sort(order[int(config['train_size']): int(config['train_size']) + int(config['eval_size'])])

        surrogate = LegacyBNNResponseSurrogate(
            hidden_dims=list(config['hidden_dims']),
            epochs=int(config['epochs']),
            batch_size=int(config['batch_size']),
            lr=float(config['lr']),
            bayesian_loss_weight=float(config['bayesian_loss_weight']),
            mc_samples=int(config['mc_samples']),
            target_mode='db',
            seed=seed,
            device=str(config['device']),
        )
        t0 = time.perf_counter()
        surrogate.fit(dataset, train_idx, epochs=int(config['epochs']))
        fit_sec = float(time.perf_counter() - t0)
        _, response_samples = surrogate.predict_s_params(
            dataset,
            eval_idx,
            mc_samples=int(config['mc_samples']),
            return_samples=True,
        )

        for local_pos, sample_idx in enumerate(eval_idx):
            true_s = dataset.s_params[int(sample_idx)]
            profile = uncertainty_profile_from_samples(response_samples, local_pos)
            for method in ['uaw', 'uniform', 'random']:
                method_rng = np.random.default_rng(seed * 100_000 + int(sample_idx) * 10 + ['uaw', 'uniform', 'random'].index(method))
                indices = select_frequency_indices(method, profile, int(config['freq_sample_size']), method_rng)
                try:
                    pred_s = reconstruct_sparams(true_s, indices)
                    status = 'ok'
                    error = None
                    s11_mae = db_mae(true_s, pred_s, 0, 0)
                    s21_mae = db_mae(true_s, pred_s, 0, 1)
                    mean_mae = float((s11_mae + s21_mae) / 2.0)
                    obj_err = float(objective_abs_error(true_s, pred_s))
                except Exception as exc:
                    status = 'failed'
                    error = f'{type(exc).__name__}: {exc}'
                    s11_mae = np.nan
                    s21_mae = np.nan
                    mean_mae = np.nan
                    obj_err = np.nan
                rows.append({
                    'seed': seed,
                    'sample_idx': int(sample_idx),
                    'case_id': dataset.case_ids[int(sample_idx)],
                    'method': method,
                    'status': status,
                    'error': error,
                    'fit_sec': fit_sec,
                    'freq_sample_size': int(len(indices)),
                    'sampled_freqs_ghz': ';'.join(f'{dataset.freqs[i] / 1e9:.6g}' for i in indices),
                    's11_db_mae': s11_mae,
                    's21_db_mae': s21_mae,
                    'mean_db_mae': mean_mae,
                    'objective_abs_error': obj_err,
                    'true_objective': objective_value(true_s),
                })

    rows_df = pd.DataFrame(rows)
    rows_df.to_csv(OUTPUT_DIR / 'reconstruction_rows.csv', index=False)

    ok = rows_df[rows_df['status'] == 'ok'].copy()
    metric_aggs = dict(
        cases=('sample_idx', 'count'),
        median_mean_db_mae=('mean_db_mae', 'median'),
        mean_mean_db_mae=('mean_db_mae', 'mean'),
        median_objective_abs_error=('objective_abs_error', 'median'),
        mean_objective_abs_error=('objective_abs_error', 'mean'),
    )
    summary_by_seed_method = (
        ok.groupby(['seed', 'method'], as_index=False)
        .agg(**metric_aggs)
        .sort_values(['seed', 'method'])
    )
    summary_by_seed_method.to_csv(OUTPUT_DIR / 'summary_by_seed_method.csv', index=False)

    summary = (
        ok.groupby(['method'], as_index=False)
        .agg(**metric_aggs)
        .sort_values('method')
    )
    summary.to_csv(OUTPUT_DIR / 'summary_overall.csv', index=False)

    pairs = []
    for metric in ['mean_db_mae', 'objective_abs_error']:
        wide = ok.pivot_table(index=['seed', 'sample_idx'], columns='method', values=metric, aggfunc='first').dropna()
        for baseline in ['uniform', 'random']:
            if {'uaw', baseline}.issubset(wide.columns):
                pairs.append({
                    'metric': metric,
                    'comparison': f'uaw_vs_{baseline}',
                    'paired_cases': int(len(wide)),
                    'uaw_win_rate': float((wide['uaw'] <= wide[baseline]).mean()) if len(wide) else np.nan,
                    'uaw_median_delta': float((wide['uaw'] - wide[baseline]).median()) if len(wide) else np.nan,
                })
    paired = pd.DataFrame(pairs)
    paired.to_csv(OUTPUT_DIR / 'paired_uaw_vs_baselines.csv', index=False)
    return rows_df, summary, paired

rows_df, summary_df, paired_df = run_validation(CONFIG)
summary_df, paired_df

## Diagnostic Plots

The plots are lightweight run diagnostics, not publication figures. They make it easy to spot whether UAW is improving the median errors and where each method sampled frequency points.

In [ ]:
import matplotlib.pyplot as plt


def _parse_sampled_freqs_ghz(value):
    if not isinstance(value, str) or not value:
        return np.asarray([], dtype=float)
    return np.asarray([float(item) for item in value.split(';')], dtype=float)


def write_diagnostic_plots(rows, summary, paired):
    plot_dir = OUTPUT_DIR / 'plots'
    plot_dir.mkdir(parents=True, exist_ok=True)
    saved = []
    method_order = ['uaw', 'uniform', 'random']

    if not summary.empty:
        plot_summary = summary.set_index('method').reindex(method_order).dropna(how='all')
        fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
        axes[0].bar(plot_summary.index, plot_summary['median_mean_db_mae'])
        axes[0].set_ylabel('Median dB MAE')
        axes[0].set_title('Full-response reconstruction')
        axes[1].bar(plot_summary.index, plot_summary['median_objective_abs_error'])
        axes[1].set_ylabel('Median objective abs error')
        axes[1].set_title('Objective preservation')
        path = plot_dir / 'summary_medians.png'
        fig.savefig(path, dpi=160)
        plt.close(fig)
        saved.append(path)

    if not paired.empty:
        fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
        labels = [f"{row.metric}\n{row.comparison}" for row in paired.itertuples()]
        ax.bar(np.arange(len(paired)), paired['uaw_win_rate'])
        ax.axhline(0.5, color='black', linestyle='--', linewidth=1)
        ax.set_xticks(np.arange(len(paired)), labels, rotation=20, ha='right')
        ax.set_ylim(0.0, 1.0)
        ax.set_ylabel('UAW paired win rate')
        path = plot_dir / 'paired_win_rates.png'
        fig.savefig(path, dpi=160)
        plt.close(fig)
        saved.append(path)

    ok = rows[rows['status'] == 'ok'].copy()
    if not ok.empty:
        first = ok[['seed', 'sample_idx']].drop_duplicates().iloc[0]
        subset = ok[(ok['seed'] == first['seed']) & (ok['sample_idx'] == first['sample_idx'])]
        fig, ax = plt.subplots(figsize=(10, 3), constrained_layout=True)
        y_lookup = {method: pos for pos, method in enumerate(method_order)}
        for _, row in subset.iterrows():
            freqs_ghz = _parse_sampled_freqs_ghz(row['sampled_freqs_ghz'])
            ax.scatter(freqs_ghz, np.full_like(freqs_ghz, y_lookup[row['method']], dtype=float), label=row['method'], s=18)
        ax.set_yticks(list(y_lookup.values()), list(y_lookup.keys()))
        ax.set_xlabel('Frequency (GHz)')
        ax.set_title(f"Sampled frequency points: seed={int(first['seed'])}, sample={int(first['sample_idx'])}")
        ax.grid(axis='x', alpha=0.25)
        ax.legend(loc='upper right')
        path = plot_dir / 'sampled_frequency_points.png'
        fig.savefig(path, dpi=160)
        plt.close(fig)
        saved.append(path)

    return [str(path.relative_to(REPO_ROOT)) for path in saved]


plot_paths = write_diagnostic_plots(rows_df, summary_df, paired_df)
plot_paths

## Quick Acceptance Check

For a full run, UAW should have no worse median full-response dB MAE and objective absolute error than both baselines, with paired win-rate above 50%. Smoke mode is only a functionality check.

In [ ]:
def acceptance_table(summary, paired):
    summary = summary.set_index('method')
    checks = []
    for metric in ['median_mean_db_mae', 'median_objective_abs_error']:
        if {'uaw', 'uniform', 'random'}.issubset(summary.index):
            checks.append({
                'check': metric,
                'passes': bool(summary.loc['uaw', metric] <= summary.loc[['uniform', 'random'], metric].min()),
                'uaw': float(summary.loc['uaw', metric]),
                'best_baseline': float(summary.loc[['uniform', 'random'], metric].min()),
            })
    for _, row in paired.iterrows():
        checks.append({
            'check': f"{row['metric']} {row['comparison']} win_rate",
            'passes': bool(row['uaw_win_rate'] > 0.5),
            'uaw': float(row['uaw_win_rate']),
            'best_baseline': 0.5,
        })
    return pd.DataFrame(checks)

acceptance_df = acceptance_table(summary_df, paired_df)
acceptance_df.to_csv(OUTPUT_DIR / 'acceptance_checks.csv', index=False)
acceptance_df